# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [13]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [16]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1985


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [22]:
simulator.get_actions(None)

[-1, 0, 1]

In [17]:
random.choice([-1, 0, 1])

0

In [25]:
actions = simulator.get_actions(None)

def random_policy(state, actions):
    return random.choice(actions)

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [27]:
import pandas as pd
log_random = []

for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
    print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
    log_random.append({
        'curr_state': curr_state,
        'curr_action': curr_action,
        'next_reward': next_reward,
        'next_state': next_state 
    })
log_random_df = pd.DataFrame(log_random)
log_random_df

CURR STATE 1989 ACTION -1 NEXT REWARD 0 NEXT STATE 1852
CURR STATE 1852 ACTION 0 NEXT REWARD 0 NEXT STATE 1724
CURR STATE 1724 ACTION -1 NEXT REWARD 0 NEXT STATE 1595
CURR STATE 1595 ACTION 0 NEXT REWARD 0 NEXT STATE 1475
CURR STATE 1475 ACTION 1 NEXT REWARD 0 NEXT STATE 1356
CURR STATE 1356 ACTION 0 NEXT REWARD 0 NEXT STATE 1228
CURR STATE 1228 ACTION 1 NEXT REWARD 0 NEXT STATE 1101
CURR STATE 1101 ACTION 0 NEXT REWARD 0 NEXT STATE 965
CURR STATE 965 ACTION 0 NEXT REWARD 0 NEXT STATE 829
CURR STATE 829 ACTION 1 NEXT REWARD 0 NEXT STATE 694
CURR STATE 694 ACTION 0 NEXT REWARD 0 NEXT STATE 686
CURR STATE 686 ACTION 1 NEXT REWARD 0 NEXT STATE 679
CURR STATE 679 ACTION -1 NEXT REWARD 0 NEXT STATE 798
CURR STATE 798 ACTION 0 NEXT REWARD 0 NEXT STATE 790
CURR STATE 790 ACTION 0 NEXT REWARD 0 NEXT STATE 782
CURR STATE 782 ACTION 1 NEXT REWARD 0 NEXT STATE 775
CURR STATE 775 ACTION 1 NEXT REWARD 0 NEXT STATE 896
CURR STATE 896 ACTION 0 NEXT REWARD 0 NEXT STATE 1024
CURR STATE 1024 ACTION -1 N

,curr_state,curr_action,next_reward,next_state
0,1989,-1,0,1852
1,1852,0,0,1724
2,1724,-1,0,1595
3,1595,0,0,1475
4,1475,1,0,1356
5,1356,0,0,1228
6,1228,1,0,1101
7,1101,0,0,965
8,965,0,0,829
9,829,1,0,694


Submit "log-random.tsv" in Gradescope.

In [28]:
log_random_df.to_csv('log-random.tsv', sep='\t', index=False)

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [62]:
alpha = 1
gamma = 0.9

In [71]:
# Table of q_values for all possible state-action pairs
n_states = simulator.resolution * simulator.resolution * len(simulator.DIRECTIONS)
q_table = pd.DataFrame([
    {'state': state, 'action': action, 'q_value': 0.0}
    for state in range(n_states)
    for action in actions
])
q_table

,state,action,q_value
0,0,-1,0.0
1,0,0,0.0
2,0,1,0.0
3,1,-1,0.0
4,1,0,0.0
...,...,...,...
6139,2046,0,0.0
6140,2046,1,0.0
6141,2047,-1,0.0
6142,2047,0,0.0


In [72]:
q_random = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        #print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
        
        # We need to find what the old q_value was for the current action
        old_q_value = q_table[
            (q_table['state'] == curr_state) &
            (q_table['action'] == curr_action)
        ]['q_value'].item()

        # Get largest q_value in next state
        next_state_rows = q_table['state'] == next_state
        max_next_q_value = q_table.loc[next_state_rows, 'q_value'].max()

        # Calculate new q_value
        new_q_value = old_q_value + alpha * (
            next_reward + gamma * max_next_q_value - old_q_value
        )

        # Update q_table
        q_table.loc[(
            (q_table['state'] == curr_state) &
            (q_table['action'] == curr_action)
        ), 'q_value'] = new_q_value

        q_random.append({
            'curr_state': curr_state,
            'curr_action': curr_action,
            'next_reward': next_reward,
            'next_state': next_state,
            'old_value': old_q_value,
            'new_value': new_q_value
        })

Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [73]:
q_random_df = pd.DataFrame(q_random)
q_random_df

,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,124,0,0,124,0.0,0.0
1,124,0,0,124,0.0,0.0
2,124,1,0,125,0.0,0.0
3,125,1,0,118,0.0,0.0
4,118,1,0,111,0.0,0.0
...,...,...,...,...,...,...
32763,376,1,0,505,0.0,0.0
32764,505,1,0,634,0.0,0.0
32765,634,1,0,635,0.0,0.0
32766,635,-1,0,506,0.0,0.0


In [74]:
q_random_df.to_csv('q-random.tsv', sep='\t', index=False)

Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [76]:
# Reset q_table
q_table = pd.DataFrame([
    {'state': state, 'action': action, 'q_value': 0.0}
    for state in range(n_states)
    for action in actions
])
q_table

,state,action,q_value
0,0,-1,0.0
1,0,0,0.0
2,0,1,0.0
3,1,-1,0.0
4,1,0,0.0
...,...,...,...
6139,2046,0,0.0
6140,2046,1,0.0
6141,2047,-1,0.0
6142,2047,0,0.0


In [109]:

def epsilon_greedy_policy(state, actions):
    # hard-code epsilon=0.25. this is high but the environment is deterministic.
    eps = 0.25

    # Pick a random float between 0-1. If less than eps pick a random action
    if random.random() < eps:
        # Random action
        return random_policy(None, actions)

    # Pick optimal policy based on q-values so far
    state_rows = q_table[q_table["state"] == state]
    best_value = state_rows["q_value"].max()
    best_actions = state_rows.loc[
        state_rows["q_value"] == best_value, "action"
    ].tolist()

    return random.choice(best_actions)

epsilon_greedy_policy(state=0, actions=simulator.get_actions(None))

1

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [111]:
q_greedy = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        #print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

        if next_reward > 0:
            # moving to terminal state
            break

        # We need to find what the old q_value was for the current action
        old_q_value = q_table[
            (q_table['state'] == curr_state) &
            (q_table['action'] == curr_action)
        ]['q_value'].item()

        # Get largest q_value in next state
        next_state_rows = q_table['state'] == next_state
        max_next_q_value = q_table.loc[next_state_rows, 'q_value'].max()

        # Calculate new q_value
        new_q_value = old_q_value + alpha * (
            next_reward + gamma * max_next_q_value - old_q_value
        )

        # Update q_table
        q_table.loc[(
            (q_table['state'] == curr_state) &
            (q_table['action'] == curr_action)
        ), 'q_value'] = new_q_value

        q_greedy.append({
            'curr_state': curr_state,
            'curr_action': curr_action,
            'next_reward': next_reward,
            'next_state': next_state,
            'old_value': old_q_value,
            'new_value': new_q_value
        })

In [112]:
q_greedy_df = pd.DataFrame(q_greedy)
q_greedy_df

,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,2046,0,0,2038,0.0,0.0
1,2038,0,0,2030,0.0,0.0
2,2030,0,0,2022,0.0,0.0
3,2022,-1,0,2013,0.0,0.0
4,2013,0,0,1877,0.0,0.0
...,...,...,...,...,...,...
29423,495,1,0,608,0.0,0.0
29424,608,-1,0,743,0.0,0.0
29425,743,0,0,863,0.0,0.0
29426,863,-1,0,982,0.0,0.0


Submit "q-greedy.tsv" in Gradescope.

In [113]:
q_greedy_df.to_csv('q-greedy.tsv', sep='\t', index=None)

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [ ]:
# YOUR CHANGES HERE

...

Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [ ]:
# YOUR CHANGES HERE

...

Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.